# סיווג אוטומטי של משלחי יד | מטלת מדען נתונים
**הלשכה המרכזית לסטטיסטיקה — הוכחת היתכנות על נתונים סינתטיים**

מטרת המטלה היא אימון מודל AI לחיזוי **SemelMishlachSofi**, הצגת חיזויים, והדגמת תהליך שניתן להריץ במלואו ב-Google Colab.

**נתונים:** 200 רשומות סקר, כפי שחולצו מהחוצץ Survey Dataset בקובץ DUMMY_DATASET.xlsx. הנתונים מוטמעים במחברת כך שהיא עצמאית ואינה דורשת העלאה ידנית או הרשאות Google Drive. חוצץ Metadata שימש להבנת השדות.

**מגבלות מפורשות במטלה:** הנתונים הם סימולציה; כ-90% מהסימולים הידניים הוערכו כנכונים; קודים המכילים X הם סימול חלקי; אין משמעות לביצועי החיזוי של הסימולציה. לכן המדדים שיוצגו הם לצורך בדיקת התקינות של תהליך האימון וההערכה בלבד.

**קוד:** [GitHub](https://github.com/shulruferor-sudo/Advance-Data-Science/blob/main/project_notebooks/survey_occupation_classification.ipynb) | [פתיחה ב-Google Colab](https://colab.research.google.com/github/shulruferor-sudo/Advance-Data-Science/blob/main/project_notebooks/survey_occupation_classification.ipynb)

**עקרונות:** הפרדה בין נתוני אימון ובדיקה, מניעת זליגת מידע, שמירת קודים כטקסט, בידוד סימול חלקי, ובקרת איכות על תוויות מתנגשות.

## 1. סביבת הרצה ותלויות
המחברת נשענת על חבילות Python המותקנות כברירת מחדל בסביבת Colab הרגילה. החבילות משמשות לעיבוד נתונים וללמידה מונחית. אין צורך בשירות API, בקובץ חיצוני או במפתח גישה.

In [ ]:
import io
import hashlib
import re
import sys
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

## 2. טעינת נתוני הסקר
העותק שלהלן נגזר מ-200 רשומות המשתתפים בקובץ המקורי. שורת הסיכום הנוספת בתחתית חוצץ הסקר אינה משתתף ולכן אינה נכללת. העמודה הטכנית של אינדקס הייצוא מוסרת; ID נשמר רק לקישור החיזוי לרשומת המקור ואינו משמש לתחזית.

הכללת הנתונים במחברת מבטיחה שניתן להריץ את כל התאים ב-Colab גם בלי קישור לקובץ הפרטי ב-Drive.

In [ ]:
SURVEY_CSV = (
    ",ID,YeshuvAvoda,ShemAvoda,SugAvoda,ShemMachlaka,SugMachlaka,EzoAvoda,TeurPeula,TeurTafkid,MenahelEtMi,MaamadAvoda,MakorSachar,TeudaGvoha,shnotlimud,Gil,TarichHathala,SemelAnafSofi,SemelMishlachSofi\n0,"
    "1,ירושלים,\"למ\"\"ס\",\"איסוף, עיבוד ופרסום נתונים\",מיקרו כלכלה,טיפול בנתוני מיקרו,סגן מנהל אגף,\"כנ\"\"ל\",סגן מנהל אגף,3,1,1,6,22,55,2020-06-15 00:00:00,8411,2120\n1,2,ירושלים,בית חולים הדסה עין כרם,טיפול בחו"
    "לים,רדיואתרפיה,טיפול בחולי סרטן,טכנאית רדיותרפיה,\"כנ\"\"ל\",\"כנ\"\"ל\",,1,1,6,22,55,2022-07-01 00:00:00,8610,3311\n2,3,באר שבע,קריית הממשלה,מתן שירות למשתמשים באתר GOV,תמיכה בלקוחות,\"כנ\"\"ל\",נותנת תמיכה ללקוח"
    "ות שמבקשים עזרה,מסבירה איך להשתמש בשירותים ממשלתיים באתר האינטרנט,,,1,1,5,15,29,2021-06-06 00:00:00,8411,3512\n3,4,תל אביב,משרד החינוך,בית ספר יסודי,שמירה,,שומר בבית הספר,,,,1,2,3,12,48,2024-07-01 00:0"
    "0:00,8010,5414\n4,5,תל אביב,אמיר כהן,מטפל בקשיש במשפחה,,,מטפל בקשיש,\"נותן אוכל, עוזר בניקיון בית, עוזר להתרחץ\",,,1,1,4,14,42,2010-01-11 00:00:00,XXXX,5322\n5,6,חיפה,משטרה,שוטר,,,שוטר,,,,1,1,4,14,35,2022"
    "-05-31 00:00:00,8423,5412\n6,7,קרית גת,אינטל,ייצור צ'יפים למחשבים,חדר נקי,,עובד בחדר נקי,מפקח על תהליך ייצור צ'יפים,,,1,1,5,15,65,1999-07-07 00:00:00,2610,3122\n7,8,חיפה,אינטל,פיתוח תוכנה לצ'יפים,הנדסת "
    "תוכנה,,\"מהנדס תוכנה, מנהל צוות עובדים\",מפתח קוד לצ'יפים,,1,1,1,6,18,55,2000-11-10 00:00:00,7201,2512\n8,9,אשקלון,שופר סל,קופאית,,,קופאית,,,,1,3,2,11,52,2022-11-11 00:00:00,4711,5230\n9,10,הרצליה,EMILIO,"
    "\"הייטק, פיתוח תוכנה \",,,מתכנת,,,,1,1,5,16,24,2021-12-14 00:00:00,6201,2514\n10,11,חדרה,בנק לאומי,ניהול כספים של אנשים,מחשוב,תמיכה במערכות מחשוב של הבנק,מתחזק רשת מחשבים,נותן תמיכה לעובדים בבנק ופוטר בע"
    "יות תקשורת,טכנאי מחשבים,,1,2,4,15,31,2022-07-01 00:00:00,6202,3513\n11,12,רעננה,בנק מזרחי,פקיד אשראי,עסקים,טיפול בלקוחות עסקיים,\"טיפול בלקוחות עסקיים של הבנק, בדיקות לצורך מתן הלוואות\",\"כנ\"\"ל\",כלכלן,,1"
    ",1,6,17,42,2013-07-11 00:00:00,6419,2631\n12,13,ירושלים,מסעדה,מסעדה,,,מלצר,,,,1,1,3,14,23,2022-08-01 00:00:00,5610,5131\n13,14,יוקנעם,נבידיה,מפעל חלקים אלקטרוניים,פס ייצור,,עובד בפס ייצור,,,,1,1,3,12,25"
    ",2015-07-10 00:00:00,2630,8212\n14,15,תל אביב,NVIDIA,פיתוח תוכנה,,,מפתח אפליקציות,,,2,1,1,4,15,27,2019-11-01 00:00:00,6203,2512\n15,16,עפולה,סופר פארם,מכירת קוסמטיקה,קוסמטיקה וטיפוח,,יועצת יופי לאנשים,מ"
    "סבירה לאנשים לגבי מוצרים של חברת LOREAL,,,1,1,3,13,32,2022-04-01 00:00:00,4772,5223\n16,17,בת ים,קוסמטיקאית עצמאית,,,,קוסמטיקאית,,,,4,,4,14,35,2024-07-31 00:00:00,9602,5142\n17,18,ירושלים,משרד ממשלתי,חס"
    "וי,חסוי,,לא יכול לפרט,,,3,1,1,6,18,45,2000-07-01 00:00:00,8411,1XXX\n18,19,ירושלים,עיריית ירושלים,גן ילדים,,,גננת לגילאים 4-6,,,,1,4,5,15,38,2011-02-01 00:00:00,8501,2342\n19,20,תל אביב,עיריית תל אביב,,"
    "מחלקת רווחה,טיפול באנשים עם בעיות,עובדת סוציאלית במחלקת רווחה,,,,1,1,6,17,46,2024-07-31 00:00:00,8890,2635\n20,21,שדרות,אוסם,ייצור פסטה,הנהלת חשבונות,חישובי שכר לעובדים,חשבת שכר לעובדים,,,,1,1,4,14,37,"
    "2022-11-01 00:00:00,1074,4313\n21,22,תל אביב,אוניברסיטת תל אביב,חוקר במעבדה,,,חוקר במעבדה ביולוגית,,,,1,1,7,20,46,2022-12-11 00:00:00,8550,2131\n22,23,משמר השרון,עובד במפעל של הקיבוץ,אפיית לחם,,,אורז לח"
    "מים לפני שליחה ללקוחות,,,,6,,3,12,78,2011-11-11 00:00:00,1071,8183\n23,24,הרצליה,חברת הייטק,פיתוח מוצרי תוכנה,,,DATA ANALYST,,,,1,1,6,18,44,2000-05-01 00:00:00,6201,2511\n24,25,ירושלים,תיאטרון ירושלים,ה"
    "פקת הצגות,רכש,הזמנת ציוד,מטפל בהזמנות של ציוד שונה להצגות,,,,1,1,4,13,31,2018-11-01 00:00:00,9000,3323\n25,26,אופקים,בית ספר תיכון,,עובד ניקיון,,מנקה בית ספר אחרי הלימודים,,,,1,2,2,10,28,2022-04-11 00:"
    "00:00,8121,9112\n26,27,מעלה החמישה,בית מלון,אירוח ,משק,,חשדרנית,,,,1,1,3,12,32,2022-07-31 00:00:00,5510,9112\n27,28,אילת,מסעדה,,,,טבח,,,,1,1,4,13,40,2000-07-15 00:00:00,5610,5120\n28,29,תל אביב,קניון עזר"
    "יאלי,חנות למכירת בגדים,,,מורכת,,,,1,1,3,12,25,2011-11-01 00:00:00,4770,5223\n29,30,חצור,גידול תפוזים ,,,,אוסף תפוזים בפרדס של קיבוץ,,,,6,,4,13,58,2024-12-31 00:00:00,123,9211\n30,31,ירושלים,סולל בונה,את"
    "ר בניה אצל קבלן,עבודות חשמל,,התקנת חשמל בדירות חדשות,עוזר לחשמלאי,,,1,2,3,12,22,2022-12-01 00:00:00,4323,7414\n31,32,ירושלים,\"למ\"\"ס\",\"איסוף, עיבוד ופרסום נתונים\",מיקרו כלכלה,טיפול בנתוני מיקרו,סגן מנהל"
    " אגף,\"כנ\"\"ל\",סגן מנהל אגף,3,1,1,6,22,55,2020-06-15 00:00:00,8411,2120\n32,33,ירושלים,בית חולים הדסה עין כרם,טיפול בחולים,רדיואתרפיה,טיפול בחולי סרטן,טכנאית רדיותרפיה,\"כנ\"\"ל\",\"כנ\"\"ל\",,1,1,6,22,55,2022-0"
    "7-01 00:00:00,8610,3211\n33,34,מודיעין,מקורות,תיקון קווי מים,תמיכה בלקוחות,\"כנ\"\"ל\",נותנת תמיכה ללקוחות שמבקשים עזרה,עוזר לתקן צינורות,,1,4,4,3,7,29,2026-06-06 00:00:00,3600,7126\n34,35,ירושלים,משרד החינ"
    "וך,בית ספר יסודי,שמירה,,שומר בבית הספר,,,,1,2,3,12,48,2024-07-01 00:00:00,8010,5414\n35,36,תל אביב,אחים כהן,מטפל בחולה במשפחה,,,מטפל בקשישים חולים,מאכיל ומנקה,,,1,1,4,14,42,2010-01-11 00:00:00,XXXX,532"
    "2\n36,37,חיפה,משטרה,שוטרת,פלילית,,שוטרת,,,,1,1,4,14,35,2020-05-31 00:00:00,8423,5412\n37,38,קרית גת,INTEL,ייצור צ'יפים ,ייצור צ'יפים ,,עובד בחדר נקי,מפקח על תהליך ייצור צ'יפים,,3,1,1,5,15,65,1999-07-07 "
    "00:00:00,2610,3122\n38,39,חיפה,אינטל,פיתוח תוכנה לצ'יפים,הנדסת תוכנה,,\"מהנדס תוכנה, מנהל צוות פיתוח\",פיתוח קוד,0,2,1,1,6,18,55,2000-11-10 00:00:00,7201,2512\n39,40,חדרה,שופרסל,סופר,,,קופאית,,,,1,3,2,11,"
    "52,2022-11-11 00:00:00,4711,5230\n40,41,הרצליה,מיקרוסופט,פיתוח,R&D,R&D,מתכנת,,,,1,1,5,16,24,2021-12-14 00:00:00,6202,2514\n41,42,חדרה,בנק לאומי,הלאות,אשראי,אשראי,בחינת יכולת החזר,בחינת יכולת החזר,כלכלן,"
    ",1,2,4,15,31,2022-07-01 00:00:00,6419,2631\n42,43,רעננה,בנק מזרחי,מתן הלוואות,פרטיים,טיפול בבקשת הלואות,\"טיפול בלקוחות עסקיים של הבנק, בדיקות לצורך מתן הלוואות\",\"כנ\"\"ל\",כלכלן,,1,1,6,17,42,2013-07-11 00"
    ":00:00,6419,2631\n43,44,ירושלים,מסעדה,מסעדה,,,שוטף כלים,,,,1,1,3,14,23,2022-08-01 00:00:00,5610,9412\n44,45,ירושלים,מסעדה,,,,,,,,,,,,,2023-09-01 00:00:00,5610,XXXX\n45,46,יוקנעם,נבידיה,מפעל חלקים אלקטרונ"
    "יים,ייצור צ'יפים ,,עובד כללי,,,,1,1,3,12,25,2015-07-10 00:00:00,2630,9329\n46,47,תל אביב,NVIDIA,פיתוח תוכנה,,,מפתח תוכנה,,,,1,1,4,15,27,2019-11-01 00:00:00,6203,2512\n47,48,תל אביב,סופר פארם,מכירת קוסמט"
    "יקה,קוסמטיקה וטיפוח,,יועצת יופי לאנשים,מסבירה לאנשים לגבי מוצרים של חברת LOREAL,,,1,1,3,13,32,2022-04-01 00:00:00,4772,5223\n48,49,בת ים,קוסמטיקאית עצמאית,,,,קוסמטיקאית,,,,4,1,4,14,35,2024-07-31 00:00:"
    "00,9602,5142\n49,50,ירושלים,משרד ממשלתי,חסוי,חסוי,,חסוי,,,3,1,1,6,18,45,2000-07-01 00:00:00,8411,1XXX\n50,51,ירושלים,עיריית ירושלים,גן ילדים,חינוך,,גננת  ,,,,1,4,5,15,38,2011-02-01 00:00:00,8501,2342\n51"
    ",52,תל אביב,עיריית תל אביב,טיפול ברווחת התושבים,מחלקת רווחה,טיפול באנשים עם בעיות,עובדת סוציאלית במחלקת רווחה,,,,1,1,6,17,46,2024-07-31 00:00:00,8890,2635\n52,53,נוף הגליל,עלית,עוגות,הנהלת חשבונות,חשבת"
    " שכר לעובדים,חישובי שכר לעובדים,חישובי שכר לעובדים,,,1,1,4,14,37,2022-11-01 00:00:00,1073,4313\n53,54,תל אביב,אוניברסיטת תל אביב,חוקר במעבדה,ביולוגיה,,חוקר במעבדה  ,,,2,1,1,7,20,46,2022-12-11 00:00:00,"
    "8550,2131\n54,55,משמר השרון,עובד במפעל  ,אריזה,אריזה,אריזה,אורז לחמים לפני שליחה ללקוחות,,,,6,,3,12,78,2011-11-11 00:00:00,1071,8183\n55,56,הרצליה,COGNYTE,מדע נתונים,R&D,,DATA SCIENTIST,בונה מודלים של A"
    "I,בונה מודלים של AI,2,1,1,6,18,44,2000-05-01 00:00:00,6201,2511\n56,57,ירושלים,תיאטרון ירושלים,שחקן,הצגות,משחק,שחקן בהצגות,,,,1,1,4,13,31,2018-11-01 00:00:00,9000,2655\n57,58,ירושלים,בית ספר תיכון,עובד "
    "ניקיון,עובד ניקיון,,מנקה בית ספר אחרי הלימודים,,,,1,2,2,10,28,2022-04-11 00:00:00,8121,9112\n58,59,מעלה החמישה,בית מלון,אירוח ,משק,,חשדרנית,,,,1,1,3,12,32,2022-07-31 00:00:00,5510,9112\n59,60,אילת,מסעדה"
    ",,,,טבח,,,,1,1,4,13,40,2000-07-15 00:00:00,5610,5120\n60,61,תל אביב,קניון עזריאלי,חנות למכירת בגדים,,,מוכרת,,,,1,1,3,12,25,2011-11-01 00:00:00,4770,5223\n61,62,מעלה החמישה,גידול תפוזים ,,,,אוסף תפוזים ב"
    "פרדס של קיבוץ,,,,6,,4,13,58,2024-12-31 00:00:00,123,9211\n62,63,ירושלים,סולל בונה,אתר בנייה ,התקנת מזגנים,,התקנת מזגנים בדירות חדשות,טכנאי מזגנים,,,1,2,3,12,22,2022-12-01 00:00:00,4327,7127\n63,64,ירושל"
    "ים,\"למ\"\"ס\",\"איסוף, עיבוד ופרסום נתונים\",מאקרו כלכלה,טיפול בנתוני מאקרו,  מנהל אגף,\"כנ\"\"ל\",סגן מנהל אגף,3,1,1,6,22,55,2020-06-15 00:00:00,8411,2120\n64,65,ירושלים,בית חולים הדסה עין כרם,טיפול בחולים,רדי"
    "ואתרפיה,טיפול בחולי סרטן,טכנאית רדיותרפיה,\"כנ\"\"ל\",\"כנ\"\"ל\",,1,1,6,22,55,2022-07-01 00:00:00,8610,3211\n65,66,אופקים,בזק,מתן שירות למשתמשים באתר GOV,תמיכה בלקוחות,\"כנ\"\"ל\",נותנת תמיכה ללקוחות שמבקשים עזרה"
    ",מסבירה איך להשתמש בשירותים ממשלתיים באתר האינטרנט,,,1,1,5,15,29,2026-06-06 00:00:00,6110,3512\n66,67,אופקים,משרד החינוך,בית ספר יסודי,שמירה,,שומר בבית הספר,,,,1,2,3,12,48,2024-07-01 00:00:00,8010,5414"
    "\n67,68,אופקים,אמיר כהן,מטפל בקשיש במשפחה,,,מטפל בקשיש,\"נותן אוכל, עוזר בניקיון בית, עוזר להתרחץ\",,,1,1,4,14,42,2010-01-11 00:00:00,XXXX,5322\n68,69,אופקים,משטרה,שוטר,,,שוטר,,,,1,1,4,14,35,2022-05-31 00"
    ":00:00,8423,5412\n69,70,אופקים,אינטל,ייצור צ'יפים למחשבים,חדר נקי,,עובד בחדר נקי,מפקח על תהליך ייצור צ'יפים,,,1,1,5,15,65,1999-07-07 00:00:00,7201,3122\n70,71,אופקים,אינטל,פיתוח תוכנה לצ'יפים,הנדסת תוכנ"
    "ה,,\"מהנדס תוכנה, מנהל צוות עובדים\",מפתח קוד לצ'יפים,,1,1,1,6,18,55,2000-11-10 00:00:00,7201,2512\n71,72,אופקים,שופר סל,קופאית,,,קופאית,,,,1,3,2,11,52,2022-11-11 00:00:00,4711,5230\n72,73,אופקים,EMILIO,\""
    "הייטק, פיתוח תוכנה \",,,מתכנת,,,,1,1,5,16,24,2021-12-14 00:00:00,6201,2514\n73,74,אילת,בנק לאומי,ניהול כספים של אנשים,מחשוב,תמיכה במערכות מחשוב של הבנק,מתחזק רשת מחשבים,נותן תמיכה לעובדים בבנק ופוטר בעי"
    "ות תקשורת,טכנאי מחשבים,,1,2,4,15,31,2022-07-01 00:00:00,6202,3513\n74,75,אילת,בנק מזרחי,מתן הלוואות,עסקים,טיפול בלקוחות עסקיים,\"טיפול בלקוחות עסקיים של הבנק, בדיקות לצורך מתן הלוואות\",\"כנ\"\"ל\",כלכלן,,1,"
    "1,6,17,42,2013-07-11 00:00:00,6419,2631\n75,76,אילת,מסעדה,מסעדה,,,מלצר,,,,1,1,3,14,23,2022-08-01 00:00:00,5610,5131\n76,77,יוקנעם,נבידיה,מפעל חלקים אלקטרוניים,פס ייצור,,עובד בפס ייצור,,,,1,1,3,12,25,201"
    "5-07-10 00:00:00,2630,8212\n77,78,יוקנעם,NVIDIA,פיתוח תוכנה,,,מפתח אפליקציות,,,2,1,1,4,15,27,2019-11-01 00:00:00,2630,2512\n78,79,אילת,סופר פארם,מכירת קוסמטיקה,קוסמטיקה וטיפוח,,יועצת יופי לאנשים,מסבירה "
    "לאנשים לגבי מוצרים של חברת LOREAL,,,1,1,3,13,32,2022-04-01 00:00:00,4772,5223\n79,80,אילת,קוסמטיקאית עצמאית,,,,קוסמטיקאית,,,,4,,4,14,35,2024-07-31 00:00:00,9602,5142\n80,81,אילת,משרד ממשלתי,חסוי,חסוי,,ל"
    "א יכול לפרט,,,3,1,1,6,18,45,2000-07-01 00:00:00,8411,1XXX\n81,82,אילת,עיריית תל אילת,גן ילדים,,,גננת לגילאים 4-6,,,,1,4,5,15,38,2011-02-01 00:00:00,8501,2342\n82,83,אילת,עיריית תל אילת,,מחלקת רווחה,טיפו"
    "ל באנשים עם בעיות,עובדת סוציאלית במחלקת רווחה,,,,1,1,6,17,46,2024-07-31 00:00:00,8890,2635\n83,84,שדרות,אוסם,ייצור פסטה,הנהלת חשבונות,חישובי שכר לעובדים,חשבת שכר לעובדים,,,,1,1,4,14,37,2022-11-01 00:00"
    ":00,1074,4313\n84,85,אשקלון,אוניברסיטת באר שבע,חוקר במעבדה,,,חוקר במעבדה ביולוגית,,,,1,1,7,20,46,2022-12-11 00:00:00,8550,2131\n85,86,אשקלון,עובד במפעל של הקיבוץ,אפיית לחם,,,אורז לחמים לפני שליחה ללקוחו"
    "ת,,,,6,,3,12,78,2011-11-11 00:00:00,1071,8183\n86,87,אשקלון,חברת הייטק,פיתוח מוצרי תוכנה,,,DATA ANALYST,,,,1,1,6,18,44,2000-05-01 00:00:00,6201,2511\n87,88,אשקלון,תיאטרון ירושלים,הפקת הצגות,רכש,הזמנת צי"
    "וד,מטפל בהזמנות של ציוד שונה להצגות,,,,1,1,4,13,31,2018-11-01 00:00:00,9000,3323\n88,89,אשקלון,בית ספר תיכון,,עובד ניקיון,,מנקה בית ספר אחרי הלימודים,,,,1,2,2,10,28,2022-04-11 00:00:00,8121,9112\n89,90,"
    "אשקלון,בית מלון,אירוח ,משק,,חשדרנית,,,,1,1,3,12,32,2022-07-31 00:00:00,5510,9112\n90,91,אשקלון,מסעדה,,,,טבח,,,,1,1,4,13,40,2000-07-15 00:00:00,5610,5120\n91,92,אשקלון,קניון עזריאלי,חנות למכירת בגדים,,,מ"
    "ורכת,,,,1,1,3,12,25,2011-11-01 00:00:00,4770,5223\n92,93,אשקלון,גידול תפוזים ,,,,אוסף תפוזים בפרדס של קיבוץ,,,,6,,4,13,58,2024-12-31 00:00:00,123,9211\n93,94,באר שבע,סולל בונה,אתר בניה אצל קבלן,עבודות ח"
    "שמל,,התקנת חשמל בדירות חדשות,עוזר לחשמלאי,,,1,2,3,12,22,2022-12-01 00:00:00,4323,7414\n94,95,באר שבע,\"למ\"\"ס\",סקרים,סקרים,איסוף נתונים בשטח,סוקר,\"כנ\"\"ל\",סוקר,3,1,1,6,22,55,2020-06-15 00:00:00,8411,4227\n"
    "95,96,ירושלים,בית חולים הדסה עין כרם,טיפול בחולים,רדיואתרפיה,טיפול בחולי סרטן,טכנאית רדיותרפיה,\"כנ\"\"ל\",\"כנ\"\"ל\",,1,1,6,22,55,2022-07-01 00:00:00,8610,3211\n96,97,באר שבע,מקורות,הספקת מים,הנדסה,\"כנ\"\"ל\",ת"
    "כנון צינורות,,,,1,1,5,16,55,1999-06-06 00:00:00,3600,2142\n97,98,באר שבע,משרד החינוך,בית ספר יסודי,שמירה,,שומר בבית הספר,,,,1,2,3,12,48,2024-07-01 00:00:00,8010,5414\n98,99,באר שבע,אחים כהן,מטפל בחולה ב"
    "משפחה,,,מטפל בקשישים חולים,מאכיל ומנקה,,,1,1,4,14,42,2010-01-11 00:00:00,XXXX,5322\n99,100,באר שבע,משטרה,שוטרת,פלילית,,שוטרת,,,,1,1,4,14,35,2020-05-31 00:00:00,8423,5412\n100,101,קרית גת,INTEL,ייצור צ'י"
    "פים ,ייצור צ'יפים ,,עובד בחדר נקי,מפקח על תהליך ייצור צ'יפים,,3,1,1,5,15,65,1999-07-07 00:00:00,2610,3122\n101,102,קרית גת,אינטל,פיתוח תוכנה לצ'יפים,הנדסת תוכנה,,\"מהנדס תוכנה, מנהל צוות פיתוח\",פיתוח קו"
    "ד,,2,1,1,6,18,55,2000-11-10 00:00:00,2610,2512\n102,103,באר שבע,שופר סל,קופאית,,,קופאית,,,,1,3,2,11,52,2022-11-11 00:00:00,4711,5230\n103,104,בת ים,EMILIO,\"הייטק, פיתוח תוכנה \",,,מתכנת,,,,1,1,5,16,24,20"
    "21-12-14 00:00:00,6201,2514\n104,105,בת ים,בנק לאומי,הלאות,אשראי,אשראי,בחינת יכולת החזר,בחינת יכולת החזר,כלכלן,,1,2,4,15,31,2022-07-01 00:00:00,6419,2631\n105,106,בת ים,בנק מזרחי,מתן הלוואות,עסקים,טיפול"
    " בלקוחות עסקיים,\"טיפול בלקוחות עסקיים של הבנק, בדיקות לצורך מתן הלוואות\",\"כנ\"\"ל\",כלכלן,,1,1,6,17,42,2013-07-11 00:00:00,6419,2631\n106,107,בת ים,מסעדה,מסעדה,,,שוטף כלים,,,,1,1,3,14,23,2022-08-01 00:00:"
    "00,5610,9412\n107,108,בת ים,מסעדה,,,,,,,,,,,,,2023-09-01 00:00:00,5610,XXXX\n108,109,הרצליה,נבידיה,מפעל חלקים אלקטרוניים,ייצור צ'יפים ,,עובד כללי,,,,1,1,3,12,25,2015-07-10 00:00:00,7201,9329\n109,110,הרצ"
    "ליה,NVIDIA,פיתוח תוכנה,,,מפתח תוכנה,,,,1,1,4,15,27,2019-11-01 00:00:00,7201,2512\n110,111,הרצליה,סופר פארם,מכירת קוסמטיקה,קוסמטיקה וטיפוח,,יועצת יופי לאנשים,מסבירה לאנשים לגבי מוצרים של חברת LOREAL,,,1"
    ",1,3,13,32,2022-04-01 00:00:00,4772,5223\n111,112,הרצליה,קוסמטיקאית עצמאית,,,,קוסמטיקאית,,,,4,1,4,14,35,2024-07-31 00:00:00,9602,5142\n112,113,הרצליה,משרד ממשלתי,חסוי,חסוי,,חסוי,,,3,1,1,6,18,45,2000-07-"
    "01 00:00:00,8411,1XXX\n113,114,חדרה,עיריית חדרה,גן ילדים,חינוך,,גננת  ,,,,1,4,5,15,38,2011-02-01 00:00:00,8501,2342\n114,115,חדרה,עיריית חדרה,טיפול ברווחת התושבים,מחלקת רווחה,טיפול באנשים עם בעיות,עובדת"
    " סוציאלית במחלקת רווחה,,,,1,1,6,17,46,2024-07-31 00:00:00,8890,2635\n115,116,נוף הגליל,עלית,עוגות,הנהלת חשבונות,חישובי שכר לעובדים,חשבת שכר לעובדים,,,,1,1,4,14,37,2022-11-01 00:00:00,1073,4313\n116,117,"
    "חדרה,מכללת חדרה,חוקר במעבדה,כימיה,,חוקר במעבדה  ,,,2,1,1,7,20,46,2022-12-11 00:00:00,8542,2113\n117,118,חדרה,עובד במפעל  ,אריזה,אריזה,אריזה,אורז לחמים לפני שליחה ללקוחות,,,,6,,3,12,78,2011-11-11 00:00:"
    "00,1071,8183\n118,119,חיפה,COGNYTE,פיתוח מוצרי תוכנה,R&D,,DATA ANALYST,,,2,1,1,6,18,44,2000-05-01 00:00:00,6201,2511\n119,120,חיפה,תיאטרון חיפה,שחקן,הצגות,משחק,שחקן בהצגות,,,,1,1,4,13,31,2018-11-01 00:0"
    "0:00,9000,2655\n120,121,חיפה,בית ספר תיכון,עובד ניקיון,עובד ניקיון,,מנקה בית ספר אחרי הלימודים,,,,1,2,2,10,28,2022-04-11 00:00:00,8121,9112\n121,122,חיפה,בית מלון,אירוח ,משק,,חשדרנית,,,,1,1,3,12,32,2022"
    "-07-31 00:00:00,5510,9112\n122,123,חיפה,מסעדה,,,,טבח,,,,1,1,4,13,40,2000-07-15 00:00:00,5610,5120\n123,124,חיפה,קניון עזריאלי,חנות למכירת בגדים,,,מוכרת,,,,1,1,3,12,25,2011-11-01 00:00:00,4770,5223\n124,1"
    "25,חיפה,גידול תפוזים ,,,,אוסף תפוזים בפרדס של קיבוץ,,,,6,,4,13,58,2024-12-31 00:00:00,123,9211\n125,126,חיפה,סולל בונה,אתר בנייה ,התקנת מזגנים,,התקנת מזגנים בדירות חדשות,טכנאי מזגנים,,,1,2,3,12,22,2022"
    "-12-01 00:00:00,4327,7127\n126,127,חיפה,משרד החינוך,בית ספר יסודי,שמירה,,שומר בבית הספר,,,,1,2,3,12,48,2024-07-01 00:00:00,8010,5414\n127,128,חיפה,א.א. סיעוד,מטפל בקשיש במשפחה,סיעוד,סיעוד,מטפל בקשיש,טיפ"
    "ול יום יומי בצרכים של הקשיש,,,1,1,4,12,42,2010-01-11 00:00:00,XXXX,5322\n128,129,מודיעין,משרד הפנים,קבלת קהל,תמיכה בלקוחות,\"כנ\"\"ל\",קבלת קהל,קבלת קהל,,,1,1,4,12,52,2022-06-06 00:00:00,8411,3354\n129,130,"
    "קריית גת,אוסם,ייצור עוגות,שיווק,שיווק עוגות,שיווק עוגות,מנהלת שיווק,,1,1,1,6,20,37,2022-11-01 00:00:00,1073,2431\n130,131,מודיעין,משטרה,שוטר,פלילית,,שוטר,,,,1,1,4,14,25,2020-05-31 00:00:00,8423,5412\n13"
    "1,132,מודיעין,משרד החינוך,בית ספר יסודי,שמירה,,שומר בבית הספר,,,,1,2,3,12,48,2024-07-01 00:00:00,8010,5414\n132,133,מודיעין,פרשקובסקי,קבלן בנייה,עבודות חשמל,,התקנת חשמל בדירות חדשות,עוזר לחשמלאי,,,1,2,"
    "3,12,22,2022-12-01 00:00:00,4323,7414\n133,134,עפולה,בית ספר תיכון,עובד ניקיון,עובד משק,,תחזוקה של בית הספר,,,,1,3,2,8,44,2022-04-11 00:00:00,8110,5153\n134,135,עפולה,סופרפארם,רוקח,רוקח,רוקח,רוקח,,,,1,1"
    ",5,16,35,2022-04-01 00:00:00,4772,2262\n135,136,קרית גת,עירייה,,מחלקת תרבות,,מארגנת אירועים,,,,1,1,6,17,44,2024-07-31 00:00:00,8312,3332\n136,137,רעננה,אמדוקס,\"הייטק, פיתוח תוכנה \",DELIVERY,,\"ר\"\"ש פיתוח"
    "\",פיתוח תכנה,פיתוח תכנה,,1,1,5,16,33,2021-12-14 00:00:00,6201,2512\n137,138,רעננה,אמדוקס,מנהל פיתוח,מחקר ופיתוח,מחקר ופיתוח,מנהל פיתוח,מנהל פיתןח של 2 מוצרים,מנהל פיתןח של 2 מוצרים,2,1,1,6,20,45,2000-0"
    "5-01 00:00:00,6201,1330\n138,139,רעננה,אמדוקס,מנהלת משאבי אנוש,HR,משאבי אנוש,מנהלת משאבי אנוש,מנהלת משאבי אנוש של עובדיםן בישראל,מנהלת משאבי אנוש של עובדיםן בישראל,3,1,1,6,20,39,2022-07-01 00:00:00,620"
    "1,1212\n139,140,רעננה,יועצת IT עצמאית,יועצת IT,,,יועצת IT,יועצת IT,יועצת IT,,4,1,4,15,32,2024-07-31 00:00:00,6202,2511\n140,141,רעננה,רשת בי פארם,רוקח,רוקח,רוקח,רוקח,,,,1,1,5,16,35,2022-04-01 00:00:00,4"
    "772,2262\n141,142,רעננה,שופרסל,עובד מחסן,מחסן,מחסן,,,,,1,3,3,12,44,2022-11-11 00:00:00,4711,XXXX\n142,143,רעננה,MICROSOFT,\"סמנכ\"\"ל שיווק\",שיווק,שיווק,\"סמנכ\"\"ל\",,\"סמנכ\"\"ל שיווק\",3,1,1,6,18,52,2021-12-14 "
    "00:00:00,6203,1221\n143,144,רעננה,אמדוקס,תמיכה במחשבים,IT,IT,ראש צוות,תמיכה במחשוב,תמיכה במחשוב,1,1,1,5,16,30,2021-12-14 00:00:00,6201,3512\n144,145,תל אביב,KPMG,יעוץ ,מדע נתונים,מדע נתונים,DATA ANALYST"
    ",,,2,1,1,6,18,55,2000-05-01 00:00:00,6921,2511\n145,146,תל אביב,תיאטרון הבימה,שחקן,הצגות,משחק,שחקן בהצגות,,,,1,1,4,13,31,2018-11-01 00:00:00,9000,2655\n146,147,תל אביב,בית ספר תיכון,עובד ניקיון,עובד ניק"
    "יון,,מנקה בית ספר אחרי הלימודים,,,,1,2,2,10,28,2022-04-11 00:00:00,8121,9112\n147,148,תל אביב,מלון דן,אירוח ,משק,,חדרנית,,,,1,1,3,12,32,2022-07-31 00:00:00,5510,9112\n148,149,תל אביב,תל אביב יפו - מסעדה"
    ",מסעדה,,,טבח,,,,1,1,4,13,40,2000-07-15 00:00:00,5610,5120\n149,150,תל אביב,קניון עזריאלי,גולברי,,,מוכרת,,,,1,1,3,12,25,2011-11-01 00:00:00,4770,5223\n150,151,משמר השרון,גידול תפוחים ,,,,אוסף תפוחים בפרד"
    "ס של קיבוץ,,,,6,,4,13,58,2024-12-31 00:00:00,124,9211\n151,152,תל אביב,סולל בונה,נהג באגר,חפירות,חפירות,חופר תעלות,חופר תעלות,,,1,2,3,12,25,2022-12-01 00:00:00,4103,8342\n152,153,תל אביב,משרד החינוך,מנה"
    "ל בית ספר יסודי,מנהל,,מנהל של בית הספר,,,3,1,1,6,16,48,2024-07-01 00:00:00,851X,1345\n153,154,תל אביב,א.א. סיעוד,מטפל בחולים,סיעוד,סיעוד,מטפל איש זקן חולה,טיפול יום יומי בצרכים של הקשיש,,,1,1,4,12,42,2"
    "010-01-11 00:00:00,XXXX,5322\n154,155,תל אביב,תיאטרון ,שחקן,הצגות,משחק,שחקן בהצגות,,,,1,1,4,13,31,2018-11-01 00:00:00,9000,2655\n155,156,תל אביב,בית ספר תיכון,עובד ניקיון,עובד ניקיון,,מנקה בית ספר אחרי "
    "הלימודים,,,,1,2,2,10,28,2022-04-11 00:00:00,8121,9112\n156,157,תל אביב,מלון אקאדיה,אירוח ,משק,,חשדרנית,,,,1,1,3,12,32,2022-07-31 00:00:00,5510,9112\n157,158,תל אביב,מסעדת עזורה,,,,טבח,,,,1,1,4,13,40,200"
    "0-07-15 00:00:00,5610,5120\n158,159,תל אביב,קניון עזריאלי,חנות למכירת בגדים,,,מוכרת,,,,1,1,3,12,25,2011-11-01 00:00:00,4770,5223\n159,160,תל אביב,אדרים,קבלן בנייה,התקנת מזגנים,,התקנת מזגנים בדירות חדשות"
    ",טכנאי מזגנים,,,1,2,3,12,22,2022-12-01 00:00:00,4327,7127\n160,161,תל אביב,משרד החינוך,בית ספר יסודי,שמירה,,שומר בבית הספר,,,,1,2,3,12,48,2024-07-01 00:00:00,8010,5414\n161,162,תל אביב,עירייה,,מחלקת רוו"
    "חה,,עובדת סוציאלית במחלקת רווחה,,,,1,1,6,17,44,2024-07-31 00:00:00,8890,2635\n162,163,באר שבע,רמי לוי,קופאי,,,קופאי,,,,1,3,2,11,55,2020-11-11 00:00:00,4711,5230\n163,164,תל אביב,סומך חייקין,יעוץ  וביקור"
    "ת,רואה חשבון,רואה חשבון,,,,,1,1,6,18,43,2000-05-01 00:00:00,6921,2411\n164,165,חיפה,עירייה,קבלת קהל,תמיכה בלקוחות,\"כנ\"\"ל\",קבלת קהל,קבלת קהל,,,1,1,4,12,52,2022-06-06 00:00:00,8311,335X\n165,166,קרית גת,י"
    "ועצץ מחשוב,,,,,,,,4,1,4,15,32,2024-07-31 00:00:00,6202,2511\n166,167,שדרות,אוסם,ייצור חטיפים,עבודות חשמל,,טיפול במערכות חשמל בצפעל,,,1,1,3,4,12,37,2022-11-01 00:00:00,1071,7414\n167,168,אשקלון,מקורות,מה"
    "נדס,תיקון תקלות,הנדסה,בונה ומתקן מתקני מים,בונה ומתקן מתקני מים,,2,1,1,6,16,44,2009-06-06 00:00:00,3600,2142\n168,169,הרצליה,בנק אוצר החייל,תמיכה במחשבים,מחשוב,תמיכה במערכות מחשוב של הבנק,מתחזק רשת מחש"
    "בים,נותן תמיכה לעובדים בבנק ופוטר בעיות תקשורת,טכנאי מחשבים,,1,2,4,15,31,2022-07-01 00:00:00,6202,3512\n169,170,חדרה,חברת הייטק,פיתוח תכנה,תכנה,R&D,מפתח,,,,1,1,6,18,33,2000-05-01 00:00:00,6201,2512\n170"
    ",171,רעננה,בנק הפועלים,מנהלת משאבי אנוש,\"כ\"\"א\",\"מחלקת כ\"\"א\",,,מנהלת משאבי אנוש,1,1,1,5,15,31,2022-07-01 00:00:00,6419,2423\n171,172,יוקנעם,4 עונות,קבלן בנייה,התקנת מטבחים,,התקנת מטבחים,,,,1,2,3,12,22,2"
    "022-12-01 00:00:00,4101,7522\n172,173,תל אביב,בנק לאומי,פקיד עובר ושב,\"עו\"\"ש\",קבלת קהל,קבלת קהל,,,,1,1,4,15,31,2022-07-01 00:00:00,6419,4211\n173,174,בת ים,משרד החינוך,בית ספר תיכון,מורה,מתמתיקה,מורה למ"
    "תמתיקה,,,,1,1,5,15,45,2019-07-01 00:00:00,852X,2330\n174,175,תל אביב,סופר פארם,פארם,קוסמטיקה וטיפוח,,יועצת יופי לנשים,,,,1,1,3,13,32,2022-04-01 00:00:00,4772,5223\n175,176,באר שבע,קוסמטיקאית עצמאית,,,,ק"
    "וסמטיקאית,,,,4,1,4,14,35,2024-07-31 00:00:00,9602,5142\n176,177,הרצליה,שופר סל,קופאי ראשי,קופה,,קופאי ראשי,,,1,1,3,3,12,52,2022-11-11 00:00:00,4711,5222\n177,178,אופקים,רמי לוי,קופאית,,,קופאית,,,,1,1,1,"
    "12,33,2023-12-14 00:00:00,4711,5230\n178,179,אילת,מלון דן,טבח,מטבח,,טבח,,,,1,1,1,12,35,2024-07-21 00:00:00,5510,5120\n179,180,חיפה,איחים חג'ג',קבלן בנייה,מכירות,,מכירת דירות,,,,1,1,3,12,44,2022-12-01 00"
    ":00:00,4101,3334\n180,181,חיפה,חנות פולגת,חנות למכירת בגדים,,,מוכרת,,,,1,1,3,12,25,2011-11-01 00:00:00,4770,5223\n181,182,חדרה,סופר פארם,מכירת קוסמטיקה,קוסמטיקה וטיפוח,קוסמטיקה,יועצת יופי  ,מסבירה לאנשי"
    "ם לגבי מוצריי יופי,,,1,1,3,13,32,2022-04-01 00:00:00,4772,5223\n182,183,הרצליה,עובד במפעל   הבקבוקים,אריזה,אריזה,אריזה,אורז בקבוקים,,,,6,,3,12,78,2011-11-11 00:00:00,3XXX,8183\n183,184,חדרה,בנק דיסקונט,"
    "השקעות,השקעות,יעוץ השקעות ללקוחות,יעוץ השקעות ללקוחות,יעוץ השקעות ללקוחות,יועץ השקעות,,1,1,5,15,35,2022-07-01 00:00:00,6419,2412\n184,185,רעננה,בנק לאומי,\"פקיד עו\"\"ש\",\"עו\"\"ש\",קבלת קהל,קבלת קהל,קבלת קהל"
    ",פקיד,,1,1,4,15,31,2022-07-01 00:00:00,6419,4211\n185,186,תל אביב,חנות בגדים של מעצבים,חנות למכירת בגדים,,,מוכרת,,,,1,1,3,12,25,2011-11-01 00:00:00,4770,5223\n186,187,בת ים,\"חח\"\"י\",תיקון קווי חשמל,תמיכה"
    " בתקלות,\"כנ\"\"ל\",נותנת תמיכה בהפשסקות חשמל,מתקן קווי חשמל,,,4,1,5,15,33,2026-06-06 00:00:00,4323,7413\n187,188,באר שבע,מוסך א.א.,חשמלאי,חשמלאות,,חשמלאי של רכבים חשמליים,תיקון מערכות חשמל וטעינה של רכבים"
    " חשמליים,,,1,3,3,12,60,2021-01-01 00:00:00,4523,7412\n188,189,הרצליה,בנק לאומי,פקיד אשראי,אשראי,בדיקת דירוג אשראי,בדיקת דירוג אשראי,בדיקת דירוג אשראי,,,1,1,5,15,31,2022-07-01 00:00:00,6419,3312\n189,190"
    ",אילת,הרודס,מציל,,,מציל בבריכה,,,,2,1,3,12,24,2022-01-11 00:00:00,5510,5419\n190,191,חיפה,משטרה,שוטר,יומנאי,,שוטר,,,,1,1,4,14,25,2020-05-31 00:00:00,8423,5412\n191,192,חיפה,מסעדת טעמים,מסעדה,,,מלצר,,,,1"
    ",1,4,13,40,2000-07-15 00:00:00,5610,5131\n192,193,רעננה,דואר ישראל,קבלת קהל,תמיכה בלקוחות,\"כנ\"\"ל\",,,,,1,1,3,7,59,1999-06-06 00:00:00,5310,XXXX\n193,194,מודיעין,מוסך אחים.,מכונאות,מכונאות,,מכונאי של רכבי"
    "ם חשמליים,תיקון מנועי חשמל וטעינה של רכבים חשמליים,,,1,3,3,12,60,2021-01-01 00:00:00,4523,7412\n194,195,אילת,ישרוטל,מלצר,מסעדה,,מלצר,,,,1,1,1,12,32,2022-04-11 00:00:00,5510,5131\n195,196,תל אביב,משטרה,ח"
    "בלן,חבלן,,חבלן,,,,1,1,4,14,25,2020-05-31 00:00:00,8423,5412\n196,197,תל אביב,מסעדת נונה,מסעדה,,,מלצר,,,,1,1,4,13,40,2000-07-15 00:00:00,5610,5131\n197,198,בת ים,קוסמטיקאית ,,,,קוסמטיקאית,,,,4,1,3,12,35,"
    "2024-07-31 00:00:00,9602,5142\n198,199,חדרה,גן חדרה ,גננת,חינוך,גננת,גננת  ,,,,1,1,4,15,38,2011-02-01 00:00:00,8501,2342\n199,200,ירושלים,\"למ\"\"ס\",סטטיסטיקה,,,סימול סקרים,קידוד ענף כלכלי ומשלח יד,,,1,1,5"
    ",15,30,2023-02-01 00:00:00,8411,4413\n"
)
source_digest = hashlib.sha256(SURVEY_CSV.encode("utf-8")).hexdigest()
survey = pd.read_csv(io.StringIO(SURVEY_CSV), dtype={"SemelMishlachSofi": "string", "SemelAnafSofi": "string"})
survey = survey.drop(columns=[c for c in survey if c.startswith("Unnamed:")], errors="ignore")
survey = survey.loc[survey["ID"].notna() & survey["SemelMishlachSofi"].notna()].copy()
survey["SemelMishlachSofi"] = survey["SemelMishlachSofi"].astype("string").str.strip()
assert len(survey) == 200 and survey["ID"].nunique() == 200
assert survey.shape[1] == 18
print("Participants:", len(survey), "| Fields:", survey.shape[1], "| Snapshot SHA-256:", source_digest[:16])
display(survey.head(4))

## 3. הבנת שדות ותנאי זמינות
הסקר מתאר מקום עבודה, פעילות מקום העבודה, מחלקה, תפקיד ופעולות המשתתף, לצד מאפיינים דמוגרפיים ותעסוקתיים.

**שדות שאינם משתתפים בחיזוי:** ID הוא מזהה טכני; SemelMishlachSofi הוא משתנה המטרה; SemelAnafSofi הוא סמל ענף סופי ולכן **אינו כלול כברירת מחדל**. אין במסמך הוכחה שהסמל הענפי הסופי זמין בזמן סיווג משלח היד. הכללתו ללא בדיקת תהליך העבודה עלולה ליצור תלות במידע שאינו זמין בזמן אמת.

**הערות איכות:** ערכי ״כנ״ל״ אינם מורחבים אוטומטית לערך אחר, משום שלא קיים כלל חד-משמעי המתיר זאת. שדות חסרים נשארים חסרים עד לתהליך העיבוד המוגדר.

In [ ]:
field_roles = pd.DataFrame([
    ("ID", "מזהה רשומה", "לא למודל"),
    ("YeshuvAvoda", "יישוב עבודה", "טקסט"),
    ("ShemAvoda", "שם מקום העבודה", "טקסט"),
    ("SugAvoda", "פעילות עיקרית של מקום העבודה", "טקסט"),
    ("ShemMachlaka", "שם מחלקה / אגף", "טקסט"),
    ("SugMachlaka", "פעילות מחלקה / אגף", "טקסט"),
    ("EzoAvoda", "העבודה בפועל", "טקסט"),
    ("TeurPeula", "פעולות עיקריות", "טקסט"),
    ("TeurTafkid", "תואר משרה", "טקסט"),
    ("MenahelEtMi", "ניהול עובדים / מנהלים", "קטגוריה"),
    ("MaamadAvoda", "מעמד תעסוקתי", "קטגוריה"),
    ("MakorSachar", "מקור שכר", "קטגוריה"),
    ("TeudaGvoha", "השכלה פורמלית", "קטגוריה"),
    ("shnotlimud", "שנות לימוד", "מספרי"),
    ("Gil", "גיל", "מספרי"),
    ("TarichHathala", "תאריך תחילת תפקיד", "שנת התחלה בלבד"),
    ("SemelAnafSofi", "סמל ענף סופי", "מוחרג: זמינות לא מוכחת"),
    ("SemelMishlachSofi", "סמל משלח יד סופי", "משתנה המטרה"),
], columns=["שדה", "משמעות לפי Metadata", "תפקיד בצינור"])
assert set(field_roles["שדה"]) == set(survey.columns)
display(field_roles)

## 4. ניתוח איכות הנתונים
קודים המכילים X מייצגים יעד שאינו מסומל באופן מלא ולכן אינם משמשים כדוגמאות אימון מפוקחות. לאחר אימון על הרשומות המלאות, נחשב עבורם חיזוי לצורך **הדגמת העברה לסקירת סמלן מקצועי**, בלי לייחס לו נכונות.

In [ ]:
label = survey["SemelMishlachSofi"]
partial_mask = label.str.contains("X", case=False, regex=False, na=False)
complete = survey.loc[~partial_mask].copy()
partial = survey.loc[partial_mask].copy()
print(f"Total: {len(survey)} | Fully coded: {len(complete)} | Partial X: {len(partial)}")
print(f"Unique observed codes: {label.nunique()} | Complete codes: {complete['SemelMishlachSofi'].nunique()}")
missingness = survey.isna().sum().sort_values(ascending=False)
display(missingness.rename("ערכים חסרים").to_frame().head(10))
counts = complete["SemelMishlachSofi"].value_counts().head(12)
ax = counts.plot.bar(figsize=(11, 4))
ax.set(xlabel="Occupation code", ylabel="Number of records",
       title="12 most frequent fully coded labels | simulated data")
plt.tight_layout()
plt.show()

## 5. קבוצות דומות, תוויות סותרות וחלוקה לאימון ובדיקה
המשתתף יכול לעדכן ולשלוח את הסקר מספר פעמים, אבל בקובץ אין מזהה משתתף הקושר גרסאות. לכן לא ניתן להבטיח זיהוי של כל גרסאות הסקר של אותו אדם. כצעד מניעתי, מגדירים קבוצת דמיון על בסיס **תיאור תעסוקה זהה לאחר נרמול רווחים** ומקצים קבוצה שלמה לאימון או לבדיקה באמצעות GroupShuffleSplit. זו קבוצת-תחליף מוגבלת, לא מזהה אמיתי.

החלוקה מתבצעת על התוויות המלאות בלבד. רמת הביצועים תלויה בכיתות שזמינות לאימון; כיתות המופיעות לראשונה בבדיקה אינן ניתנות לחיזוי כמחלקות שלמות באמצעות מסווג רב-מחלקתי רגיל.

In [ ]:
TEXT_FIELDS = [
    "YeshuvAvoda", "ShemAvoda", "SugAvoda", "ShemMachlaka",
    "SugMachlaka", "EzoAvoda", "TeurPeula", "TeurTafkid"
]
OCCUPATION_TEXT_FIELDS = TEXT_FIELDS[1:]
CATEGORY_FIELDS = ["MenahelEtMi", "MaamadAvoda", "MakorSachar", "TeudaGvoha"]
NUMERIC_FIELDS = ["shnotlimud", "Gil", "start_year"]

def clean_text(value):
    if pd.isna(value):
        return ""
    value = re.sub(r"\s+", " ", str(value).strip())
    return "" if value in {'כנ"ל', "כנ״ל"} else value

def description_group(row):
    return "|".join(clean_text(row[c]).lower() for c in OCCUPATION_TEXT_FIELDS)

def prepare_features(frame):
    features = pd.DataFrame(index=frame.index)
    features["job_text"] = frame.apply(
        lambda row: " | ".join(
            f"{field}: {clean_text(row[field])}"
            for field in TEXT_FIELDS if clean_text(row[field])
        ), axis=1
    )
    for field in CATEGORY_FIELDS:
        features[field] = (
            pd.to_numeric(frame[field], errors="coerce")
              .astype("Int64").astype("string").fillna("missing").astype(str)
        )
    for field in NUMERIC_FIELDS[:-1]:
        features[field] = pd.to_numeric(frame[field], errors="coerce")
    features["start_year"] = pd.to_datetime(
        frame["TarichHathala"], errors="coerce"
    ).dt.year
    return features

all_groups = complete.apply(description_group, axis=1)
group_counts = Counter(all_groups)
conflict_count = (
    complete.assign(description_group=all_groups)
    .groupby("description_group")["SemelMishlachSofi"]
    .nunique().gt(1).sum()
)
print("Groups:", len(group_counts),
      "| Repeated groups:", sum(size > 1 for size in group_counts.values()),
      "| Groups with conflicting labels:", conflict_count)

X = prepare_features(complete)
y = complete["SemelMishlachSofi"].astype(str)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.22, random_state=RANDOM_STATE)
idx_train, idx_test = next(splitter.split(X, y, groups=all_groups))
X_train, X_test = X.iloc[idx_train], X.iloc[idx_test]
y_train, y_test = y.iloc[idx_train], y.iloc[idx_test]
assert not set(all_groups.iloc[idx_train]).intersection(all_groups.iloc[idx_test])
unseen_test = ~y_test.isin(set(y_train))
print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("Classes in training:", y_train.nunique(),
      "| Test rows with unseen classes:", int(unseen_test.sum()))

## 6. מודל והשוואה לקו בסיס
צינור scikit-learn משלב TF-IDF ברמת תווים לטקסט עברי, OneHotEncoder למשתנים קטגוריאליים והשלמה/תקנון של משתנים מספריים. לאחר מכן LogisticRegression מבצע סיווג רב-מחלקתי. **כל רכיב הלומד פרמטרים מתאים את עצמו לנתוני האימון בלבד**.

DummyClassifier המנבא את מחלקת האימון הנפוצה הוא קו בסיס למבחן התקינות. לא מתבצעת התאמת היפר-פרמטרים על קבוצת הבדיקה.

In [ ]:
preprocessing = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(
            analyzer="char", ngram_range=(2, 5), max_features=12000,
            sublinear_tf=True
        ), "job_text"),
        ("categories", OneHotEncoder(handle_unknown="ignore"), CATEGORY_FIELDS),
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), NUMERIC_FIELDS),
    ],
    sparse_threshold=1.0,
)
model = Pipeline([
    ("preprocessing", preprocessing),
    ("classifier", LogisticRegression(
        C=2.0, max_iter=1200, solver="lbfgs"
    )),
])
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
model.fit(X_train, y_train)
y_baseline = baseline.predict(X_test)
y_pred = model.predict(X_test)
predicted_probabilities = model.predict_proba(X_test)

def metric_row(name, prediction):
    return {
        "שיטה": name,
        "accuracy": accuracy_score(y_test, prediction),
        "macro_f1": f1_score(y_test, prediction, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_test, prediction, average="weighted", zero_division=0),
    }

metrics = pd.DataFrame([
    metric_row("DummyClassifier", y_baseline),
    metric_row("TF-IDF + LogisticRegression", y_pred),
])
display(metrics.round(3))
print(f"Validation: grouped holdout, {len(X_test)} rows, "
      f"{int(unseen_test.sum())} unseen-label rows.")
print("These scores are descriptive for synthetic data only.")

## 7. תוצאות החיזוי והסבר מוגבל
התוצאות מציגות קוד מטרה מלא מול הקוד שחזה המודל. השדה model_score הוא ערך predict_proba **שאינו מכויל** ואסור לפרשו כהסתברות אמינה או כבסיס לסף אוטומציה תפעולי. המדדים אינם מדווחים על דיוק סיווג בפועל באוכלוסיית הסקרים.

In [ ]:
prediction_table = pd.DataFrame({
    "ID": complete.iloc[idx_test]["ID"].to_numpy(),
    "actual": y_test.to_numpy(),
    "predicted": y_pred,
    "correct": (y_test.to_numpy() == y_pred),
    "label_seen_in_training": (~unseen_test).to_numpy(),
    "model_score_uncalibrated": predicted_probabilities.max(axis=1).round(3),
})
display(prediction_table.head(20))
print("Held-out prediction rows:", len(prediction_table))
print("Classification report for evaluation diagnostics:")
print(classification_report(y_test, y_pred, zero_division=0))

## 8. טיפול במקרים שהסימול הידני שלהם חלקי
בכל אחד מהמקרים בעלי X אין תווית מלאה לאימון או להערכת נכונות. המודל רשאי להציע קוד לבדיקה מקצועית בלבד. אין להחליף את הסמל המקורי ללא החלטת סמלן אנושי וללא מידע נוסף.

In [ ]:
X_partial = prepare_features(partial)
partial_proposal = model.predict(X_partial)
partial_score = model.predict_proba(X_partial).max(axis=1)
manual_review = pd.DataFrame({
    "ID": partial["ID"].to_numpy(),
    "original_partial_code": partial["SemelMishlachSofi"].astype(str).to_numpy(),
    "suggested_code": partial_proposal,
    "model_score_uncalibrated": np.round(partial_score, 3),
    "route": "manual_review",
})
display(manual_review)
assert len(manual_review) == 9

## 9. תובנות ומגבלות מחקריות
- 200 רשומות אינן מספיקות להסקת ביצועי פריסה עבור מגוון גדול של סמלי משלח יד. המחלקות נדירות, וחלקן מופיעות פעם אחת בלבד.
- קודים חלקיים, כפילויות ותוויות סותרות מחייבים תהליך ניקוי והכרעת סמלנים מקצועיים לפני אימון על נתוני אמת.
- GroupShuffleSplit מונע מעבר של תיאורים זהים בין אימון לבדיקה, אבל בהיעדר מזהה משתתף או גרסת שאלון הוא **לא מבטיח** העדר דליפה בין גרסאות דומות.
- סמלים סופיים המתקבלים בתהליך עבודה אחר, למשל SemelAnafSofi, ייכללו רק אם יוכח שהם זמינים בזמן החיזוי.
- אין להשתמש בהסתברויות הלא-מכוילות לקבלת החלטות. לשם אוטומציה נדרש סט תיוג מוסכם, כיול על נתונים בלתי תלויים, בדיקות הוגנות, מנגנון הימנעות ובקרה אנושית.

## 10. מסלול הטמעה מוצע
1. **תשתית מידע:** מזהה משתתף, גרסת סקר, חותמת זמן של כל שידור, זמינות כל תכונה בזמן אמת, ושימור היסטוריית תיקונים.
2. **איכות תוויות:** מדגם adjudication בין סמלנים, ניתוח מחלוקות והכרעה בקודים חלקיים.
3. **הערכה אמינה:** נתוני אמת עדכניים, חלוקה לפי משתתף ולפי זמן, ובדיקת קבוצות/מחלקות נדירות.
4. **מודל ושירות:** צינור אימון מנוהל, חוזה נתונים, ניטור drift, רישום גרסאות ובדיקות תקינות.
5. **הטמעה הדרגתית:** הצעת קוד לסמלן לפני אוטומציה, תיעוד החלטות אנושיות, ספי איכות/הימנעות שנקבעים על אימות בלתי תלוי.

### מקורות תכנון
- [scikit-learn — Common pitfalls and data leakage](https://scikit-learn.org/stable/common_pitfalls.html)
- [scikit-learn — GroupShuffleSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html)
- [scikit-learn — ColumnTransformer and mixed types](https://scikit-learn.org/stable/auto_examples/compose/plot_column_transformer_mixed_types.html)
- [scikit-learn — TF-IDF](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)
- [scikit-learn — Probability calibration](https://scikit-learn.org/stable/modules/calibration.html)
- [Google Colab — FAQ](https://research.google.com/colaboratory/faq.html)

**ייחוס קוד:** קוד המחברת נכתב באופן עצמאי למטלה זו. מאגר Ensembles הישן שימש אך ורק להתרשמות מפרקטיקות הנדסיות, לא כמקור קוד להעתקה.